In [1]:
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets
import numpy as np
# import pandas as pd
import matplotlib.pyplot as plt
import cv2
import skimage

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading

In [2]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [3]:
#%% Load OCT study information
#folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')
# this is also folder name

# strip 14
#study_name = 'GHL_pyapp_20250326T1416'
#study_name = 'GHL_pyapp_20250327T1450'
#study_name = 'GHL_pyapp_20250415T1332'
study_name = 'GHL_pyapp_20250415T1320'

octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(study_name,folder_octexport_root);

STUDY: GHL_pyapp_20250415T1320
{   'study_has_an_ini_file': False,
    'study_has_json_info_file': False,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 20,
    'study_num_vtk_files': 0}


In [4]:
octstudy.num_oct_files

20

In [5]:
octstudy

<OCT_Study_Folder Object>
GHL_pyapp_20250415T1320 in folder D:/SGProjects/NAATOS/OCTlocal
{   'study_has_an_ini_file': False,
    'study_has_json_info_file': False,
    'study_has_yat_log': False,
    'study_num_jpg_files': 0,
    'study_num_oct_files': 20,
    'study_num_vtk_files': 0}

# Load all OCT files

In [6]:
octstudy.load_all_octs(make_pv_volume=True);    # we will use the pyvista volumes a bit later for debuggin

Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0

In [7]:
# export individual vtk files
if False:
    for count,octdata in enumerate(octstudy.octdatalist):
        octdata.pvvol.save(octstudy.folder_study_processed/'{:s}_saveasvtk_{:04d}.vtk'.format(octstudy.study_name,count));

# Stack And Merge All OCT Volumes To A VTK using PYVISTA

In [8]:
mergedvol = octstudy.pv_stack_all_volumes_along_dimension(dimension=1);

SingleFOV --> Image Dims:(690, 150, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)
Combined --> Image Dims:(690, 3000, 450) PixelSpacing[mm]:(0.003475, 0.02, 0.02)


In [9]:
print(mergedvol)

ImageData (0x24e7b065f00)
  N Cells:      927773639
  N Points:     931500000
  X Bounds:     0.000e+00, 2.394e+00
  Y Bounds:     0.000e+00, 5.998e+01
  Z Bounds:     0.000e+00, 8.980e+00
  Dimensions:   690, 3000, 450
  Spacing:      3.475e-03, 2.000e-02, 2.000e-02
  N Arrays:     1


In [10]:
octstudy.folder_study_processed.mkdir(exist_ok=True);
mergedvol.save(octstudy.folder_study_processed/'{:s}_STACKED.vtk'.format(octstudy.name));

# Process Camera RGB Images

In [ ]:
if(octstudy.num_oct_files>=1):
    octstudy.folder_study_processed.mkdir(exist_ok=True);
    tmparrays = [];
    for count,octdata in enumerate(sorted(octstudy.octdatalist, key= lambda x: int(x.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text))):
        octdata_study = octdata.cfg_oct_xml.Ocity.MetaInfo.Study.etElem.text;
        octdata_timestr = time.strftime('%Y%m%dT%H%M%S',time.gmtime(int(octdata.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text)));
        
        #fname = '{:s}_{:02d}_{:s}'.format(octdata_study,count,octdata_timestr);
        fname = '{:s}_videocamera_{:04d}'.format(octdata_study,count);
        print(fname,octdata_timestr);

        # save a .png in the processing folder
        #octdata.image.save(octstudy.folder_study_processed/(fname+'.png'))


        # # simple ITK
        # simgcam = sitk.GetImageFromArray(np.array(octdata.image)[:,:,0:3],isVector=True);

        # # camerascalingx and camerascalingy are defined as pixels/mm in the OCT probe .ini file
        # simgcam.SetSpacing(( 1/float(octdata.cfg_oct_probe['camerascalingx']), 1/float(octdata.cfg_oct_probe['camerascalingy'])));

        # # write nrrd image
        # writer = sitk.ImageFileWriter()
        # fname = folder_study_processed/'{:s}_videocamera_{:04d}.nrrd'.format(study_name,count)
        # writer.SetFileName(fname);
        # writer.Execute(simgcam);

        # # load image in slicer
        # slicerio.server.file_load(fname)
        
        # # delete image - # this is kinda important because slicer will assume files with _0001 suffixes all need to be loaded as zslices!
        # Path(fname).unlink();

In [ ]:
px_per_mm_X = float(octdata.cfg_oct_probe['camerascalingx']);
px_per_mm_Y = float(octdata.cfg_oct_probe['camerascalingy']);
print('millimeters Per Pixel X:{:} Y:{:}'.format(px_per_mm_X,px_per_mm_Y))
# vol_dimensions = (
#     int(octdata.cfg_oct_xml.Ocity.Image.SizePixel.etElem[0].text),
#     int(octdata.cfg_oct_xml.Ocity.Image.SizePixel.etElem[1].text),
#     int(octdata.cfg_oct_xml.Ocity.Image.SizePixel.etElem[2].text),
# );
# vol_spacing_mm = (
#     float(octdata.cfg_oct_xml.Ocity.Image.PixelSpacing.etElem[0].text),
#     float(octdata.cfg_oct_xml.Ocity.Image.PixelSpacing.etElem[1].text),
#     float(octdata.cfg_oct_xml.Ocity.Image.PixelSpacing.etElem[2].text),
# );
# print()
oct_scan_mm_center_X = float(octdata.cfg_oct_xml.Ocity.Image.CenterX.etElem.text) 
oct_scan_mm_center_Y = float(octdata.cfg_oct_xml.Ocity.Image.CenterY.etElem.text)
oct_scan_mm_size_X = float(octdata.cfg_oct_xml.Ocity.Image.SizeReal.etElem[1].text) # 1 = X
oct_scan_mm_size_Y = float(octdata.cfg_oct_xml.Ocity.Image.SizeReal.etElem[2].text) # 2 = Y
print('OCT Scan Center Was @ [mm]:',oct_scan_mm_center_X,oct_scan_mm_center_Y)
print('OCT Scan FOV    Was @ [mm]:',oct_scan_mm_size_X,oct_scan_mm_size_Y)

camera_scan_px_X = px_per_mm_X*oct_scan_mm_size_X;
camera_scan_px_Y = px_per_mm_Y*oct_scan_mm_size_Y;
print('Video Camera Scan Pixel Width  [px]:',camera_scan_px_X);
print('Video Camera Scan Pixel Height [px]:',camera_scan_px_Y);

camera_scan_center_px_X = px_per_mm_X*oct_scan_mm_center_X;
camera_scan_center_px_Y = px_per_mm_Y*oct_scan_mm_center_Y;
print('Video Camera Scan Pixel Center X [px]:',camera_scan_center_px_X);
print('Video Camera Scan Pixel Center Y [px]:',camera_scan_center_px_Y);


In [ ]:

imgarr = np.array(octstudy.octdatalist[10].image)[:,:,0:3];
plt.imshow(imgarr[:,:,:])
print(imgarr.shape)


In [ ]:

#x_slice = slice(imgarr.shape[0]//2)
slice_x = slice( imgarr.shape[0]//2+0, imgarr.shape[0]//2+imgarr.shape[0]//2);
plt.imshow(imgarr[slice_x,:,:])


In [ ]:
px_origin_x = imgarr.shape[1]//2-int(camera_scan_center_px_X);
px_origin_y = imgarr.shape[0]//2-int(camera_scan_center_px_Y);

camera_x_slice = slice(px_origin_x-(camera_scan_px_X/2) , px_origin_x+(camera_scan_px_X/2))
camera_y_slice = slice(px_origin_y-(camera_scan_px_Y/2) , px_origin_y+(camera_scan_px_Y/2))
print(' SliceX  :',camera_x_slice)
print(' SliceY  :',camera_y_slice)

camera_x_sliceint = slice(int(camera_x_slice.start),int(camera_x_slice.stop));
camera_y_sliceint = slice(int(camera_y_slice.start),int(camera_y_slice.stop));
print('SliceXint:', camera_x_sliceint )
print('SliceYint:', camera_y_sliceint )

plt.imshow(imgarr[ camera_y_sliceint , camera_x_sliceint , : ])

In [ ]:
if(octstudy.num_oct_files>=1):
    octstudy.folder_study_processed.mkdir(exist_ok=True);
    tmparrays = [];
    for count,octdata in enumerate(sorted(octstudy.octdatalist, key= lambda x: int(x.cfg_oct_xml.Ocity.Acquisition.Timestamp.etElem.text))):
        imgarr = np.array(octdata.image)[:,:,0:3];
        # slice using pre-determined areas corresponding to the oct scan
        tmparrays.append(imgarr[ camera_y_sliceint , camera_x_sliceint , : ]);


In [ ]:
# write image montage to .png in processed folder
img_camera_montage = Image.fromarray(np.concatenate(tmparrays,axis=1))
img_camera_montage.save((octstudy.folder_study_processed/('{:s}_montaged.png'.format(octstudy.name))))

In [ ]:
octstudy

# Processing Step 0 - Replace VTK scalars with scaled bytes

In [10]:
# make vedo volume
vdvol = vedo.Volume(mergedvol);

In [11]:
# restrict range and re-scale, cast to integer (will reduce memory by 4x)
oct_scalar_min = 30;
oct_scalar_max = 60;
scalars_rescaled_as_int = skimage.util.img_as_ubyte( (np.clip(vdvol.dataset.active_scalars,a_min=oct_scalar_min,a_max=oct_scalar_max)-oct_scalar_min)/(oct_scalar_max-oct_scalar_min) )

vdvol.dataset['OCTintensity'] = scalars_rescaled_as_int;

# Processing Step 1 - Use top-down view to draw the centerline of the wax valve

In [ ]:
print(vdvol)

In [ ]:
pl = vedo_plotters.SimonSlicer3DPlotter(
    vdvol,
    #cmaps=("gist_ncar_r", "jet", "Spectral_r", "hot_r", "bone_r"),
    cmaps=("grey","gist_ncar_r"),
    use_slider3d=False,
    show_histo=False,
    show_icon=False,
    bg="black",
    bg2="gray5",
    slice_X=True,
    slice_Y=False,
    slice_Z=False,
    scalar_range=(30.0,50.0), # dB range from OCT scalars
)

bnds = vdvol.bounds();

#vdLine = vedo.Line(p0=(0,0,bnds[5]/2),p1=(0,bnds[3],bnds[5]/2),closed=False,lw=6,c='red');
try:
    step1info
    # variable is defined
    pts = [step1info['wax_valve_manual_nodes_mm'][0], step1info['wax_valve_manual_nodes_mm'][1]];
except NameError:
    # variable was not defined
    pts = [(0, 0, bnds[5]/2) , (0, bnds[3], bnds[5]/2)];
#pl.add(vdLine);
# Add the spline tool using the same points and interact with it
sptool = pl.add_spline_tool(pts, pc='red', lw=3, closed=False);

# # Add a callback to print some info as the lines is changed
# sptool.add_observer(
#     "end of interaction", 
#     lambda o, e: (
#         print(f"Points changed! Pts= {sptool.nodes()}"),
#     )
# )

pl.parallel_projection(True); # orthographic

# # Can now add any other vedo object to the Plotter scene:
pl += vedo.Text2D(
"""STEP 1 - Find wax stripe

Instructions:

#1. select a slice pixel position using the red slider, if the wax valve region is not visible

#2. drag handles of the line to align with ends of wax valve, and centered along the wax valve

#3. when done press ESC
"""
)

pl.show(
    viewup=[0,0,-1],
    interactive=True,axes=4
);
pl.close()

step1info = dict(
    longitudinal_depth_slicer_voxels=pl.xslider.value,
    wax_valve_manual_nodes_mm=sptool.nodes(),
    longitudinal_depth_slicer_bounds_mm=pl.xslice.bounds(),
)

In [ ]:
print('Step 1 Info:')
pp.pprint(step1info);

# Step 2 - Find Other Dimension

In [ ]:
import math
# origin
slice_origin=(
    step1info['longitudinal_depth_slicer_bounds_mm'][0] ,
    (step1info['wax_valve_manual_nodes_mm'][1][1]-step1info['wax_valve_manual_nodes_mm'][0][1])/2 + step1info['wax_valve_manual_nodes_mm'][0][1] ,
    (step1info['wax_valve_manual_nodes_mm'][1][2]-step1info['wax_valve_manual_nodes_mm'][0][2])/2 + step1info['wax_valve_manual_nodes_mm'][0][2] ,
)
slice_origin = tuple([x.item() for x in slice_origin])
slice_origin

In [ ]:
#v1 = []
v1 = np.diff( np.array( step1info['wax_valve_manual_nodes_mm'] ) ,axis=0)[0];
v2 = np.array([0,1,0]);
v1_u = v1/np.linalg.norm(v1);
v2_u = v2/np.linalg.norm(v2);
print(v1_u)
print(v2_u)
#np.linalg.norm(np.array(step1info['wax_valve_manual_nodes_mm']),axis=1).T
angle_to_y_radians = math.acos(np.dot(v1_u,v2_u))
print('Degrees:',math.degrees(angle_to_y_radians))


def rotate_vector(v, axis, theta):
    """
    Rotates a vector v around an axis by an angle theta.

    Parameters:
    v (numpy.ndarray): The vector to rotate.
    axis (numpy.ndarray): The axis to rotate around.
    theta (float): The angle of rotation in radians.

    Returns:
    numpy.ndarray: The rotated vector.
    """
    axis = axis / np.linalg.norm(axis)  # Normalize the axis
    a = np.cos(theta / 2.0)
    b, c, d = -axis * np.sin(theta / 2.0)
    aa, bb, cc, dd = a * a, b * b, c * c, d * d
    bc, ad, ac, ab, bd, cd = b * c, a * d, a * c, a * b, b * d, c * d
    rotation_matrix = np.array([[aa + bb - cc - dd, 2 * (bc + ad), 2 * (bd - ac)],
                                [2 * (bc - ad), aa + cc - bb - dd, 2 * (cd + ab)],
                                [2 * (bd + ac), 2 * (cd - ab), aa + dd - bb - cc]])
    return np.dot(rotation_matrix, v)

slice_normal = rotate_vector( np.array([0,0,1]) , axis=np.array([1,0,0]), theta=angle_to_y_radians)
print('Normal Vector',slice_normal)

In [ ]:
#slice = vol.copy();
#slice_origin=( step1info['longitudinal_depth_slicer_bounds_mm'][0] , (step1info['wax_valve_manual_nodes_mm'][1][1]-step1info['wax_valve_manual_nodes_mm'][0][1])/2 , (step1info['wax_valve_manual_nodes_mm'][1][2]-step1info['wax_valve_manual_nodes_mm'][0][2])/2 )
vdPlane = vedo.Plane(pos=slice_origin,normal=slice_normal,s=(v1[1],6),c='red7',alpha=0.75)

pl = vedo_plotters.SimonSlicer3DPlotter(
    vdvol,
    #cmaps=("gist_ncar_r", "jet", "Spectral_r", "hot_r", "bone_r"),
    cmaps=("grey","gist_ncar_r"),
    use_slider3d=False,
    show_histo=False,
    show_icon=False,
    bg="black",
    bg2="gray5",
    slice_X=True,
    slice_Y=False,
    slice_Z=False,
    scalar_range=(30.0,50.0), # dB range from OCT scalars
)
#pl = vedo.Plotter();

#pl.add(vdPlane);
vdSlice = vdvol.slice_plane(origin=slice_origin,normal=slice_normal,autocrop=True).cmap('jet',vmin=38.0,vmax=60.0);
pl.add(vdSlice);

# # Can now add any other vedo object to the Plotter scene:
pl += vedo.Text2D(
"""STEP 2 - View Slice Down Center Of Wax Valve

Instructions:

#1. view

#2. when done, press ESC
"""
)

pl.show(
    viewup=[-1,0,0],
    interactive=True,axes=4
);
pl.close()

# Step 3 - Set A Crop Box

In [ ]:
slice_origin=(
    step1info['longitudinal_depth_slicer_bounds_mm'][0] ,
    (step1info['wax_valve_manual_nodes_mm'][1][1]-step1info['wax_valve_manual_nodes_mm'][0][1])/2 + step1info['wax_valve_manual_nodes_mm'][0][1] ,
    (step1info['wax_valve_manual_nodes_mm'][1][2]-step1info['wax_valve_manual_nodes_mm'][0][2])/2 + step1info['wax_valve_manual_nodes_mm'][0][2] ,
)
slice_origin = tuple([x.item() for x in slice_origin])
slice_origin

In [ ]:
# show in pyvista previously sliced vedo plot

pl = pv.Plotter(notebook=False);
pl.background_color='gray'

# add data to plotter
pl.add_mesh(pv.wrap(vdSlice.dataset),cmap='jet',clim=[40,55])

# add interactive widgets
def cb_box(evt):
    print('cb_box',evt)

wdgt = pl.add_box_widget(cb_box, rotation_enabled=False,use_planes=True,factor=1.0,color='red',outline_translation=True);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='red',normal_rotation=False,factor=0.5,origin=[2,0,0]);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='green',normal_rotation=False,factor=0.5,origin=[1,0,0]);
#wdgt = pl.add_camera_orientation_widget();
#wdgt = pl.add_box_widget()

# formatting
pl.add_box_axes();
pl.show_bounds();

pl.view_isometric();
pl.view_vector([0,0,1]);
#pl.disable();

pl.show();

In [ ]:
vdSlice = vdvol.slice_plane(origin=slice_origin,normal=slice_normal,autocrop=True).cmap('jet',vmin=38.0,vmax=60.0);
#pvSliceLong = pv.wrap(vdSlice.dataset);

slice_inplane_origin = ( (vdvol.bounds()[1]-vdvol.bounds()[0])/2 , (vdvol.bounds()[3]-vdvol.bounds()[2])/2 , (vdvol.bounds()[5]-vdvol.bounds()[4])/2 );
vdSliceLong = vdvol.slice_plane(origin=slice_inplane_origin,normal=(1,0,0),autocrop=True).cmap('jet',vmin=38.0,vmax=60.0);
#pvSliceLong = pv.wrap(vdSlice.dataset);


pl = pv.Plotter(notebook=False);
pl.background_color='gray'

# add data to plotter
pl.add_mesh(pv.wrap(vdSlice.dataset),cmap='jet',clim=[40,55])
pl.add_mesh(pv.wrap(vdSliceLong.dataset),cmap='jet',clim=[40,55])

# add interactive widgets
def cb_box(evt):
    print('cb_box',evt)


bounds = list(mergedvol.bounds)
# y longitudinal
bounds[2] = step1info['wax_valve_manual_nodes_mm'][0][1].item()
bounds[3] = step1info['wax_valve_manual_nodes_mm'][1][1].item()
# z transverse
bounds[4] = step1info['wax_valve_manual_nodes_mm'][0][2].item()
bounds[5] = step1info['wax_valve_manual_nodes_mm'][1][2].item()
bounds
wdgt = pl.add_box_widget(cb_box, bounds=bounds, rotation_enabled=False,use_planes=True,factor=1.0,color='red',outline_translation=True);
wdgt.SetHandleSize(0.01);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='red',normal_rotation=False,factor=0.5,origin=[2,0,0]);
#wdgt = pl.add_plane_widget(cb_box,normal='x',color='green',normal_rotation=False,factor=0.5,origin=[1,0,0]);
#wdgt = pl.add_camera_orientation_widget();
#wdgt = pl.add_box_widget()

# rotate the box
# the_box = pv.PolyData()
# wdgt.GetPolyData(the_box)
# the_box.rotate_y(45);
# wdgt.Set
vtkTransform = pv._vtk.vtkTransform()
vtkTransform.RotateX(math.degrees(angle_to_y_radians))
print(vtkTransform.GetMatrix())
wdgt.SetTransform(vtkTransform)


# formatting
pl.add_box_axes();
pl.show_bounds();

#pl.view_isometric();

#pl.enable_parallel_projection()
#pl.enable_image_style()

#pl.enable_parallel_projection();
#pl.disable();

def key_press_callback(key):
    if key == 'w':  # Example: Move forward
        pl.camera.zoom(1.1)  # Zoom in
    elif key == 's':  # Example: Move backward
        pl.camera.zoom(0.9)  # Zoom out
    elif key == 'a':  # Example: Pan left
        #pl.camera.azimuth = pl.camera.azimuth - 10
        pt = list(pl.camera.focal_point);
        pt[1]-=1;
        pl.camera.focal_point = pt;
    elif key == 'd':  # Example: Pan right
        #pl.camera.azimuth = pl.camera.azimuth + 10
        pt = list(pl.camera.focal_point);
        pt[1]+=1;
        pl.camera.focal_point = pt;
    pl.render() # Render the scene after camera changes
# #pl.add_callback('KeyPressEvent', key_press_callback)
#pl.add_key_event(key='a',callback=lambda *_: pl.camera.azimuth += -10)    # pan left
#pl.add_key_event(key='d',callback=lambda *_: pl.camera.azimuth += 10)    # pan right
pl.add_key_event(key='w',callback= lambda: key_press_callback("w") );
pl.add_key_event(key='s',callback= lambda: key_press_callback("s") );
pl.add_key_event(key='a',callback= lambda: key_press_callback("a") );
pl.add_key_event(key='d',callback= lambda: key_press_callback("d") );
#pl.ren_win.AddObserver('KeyPressEvent', key_press_callback);

pl.view_vector([0,0,1]);
#pl.view_yx();
#pl.camera.roll = 90
#pl.camera.azimuth = 180

pl.show();


In [ ]:
the_box = pv.PolyData()
wdgt.GetPolyData(the_box)
print(the_box)
the_box.bounds

In [ ]:
vtkTransform = pv._vtk.vtkTransform()
wdgt.GetTransform(vtkTransform)
print(vtkTransform.GetMatrix())
mtx = vtkTransform.GetMatrix()
dir(mtx)
#mtx.GetData()
vedo.vtk2numpy(vtkTransform)

In [ ]:
step3info = dict(
    box_bounds=the_box.bounds,
    box_transform_4x4=vedo.vtk2numpy(vtkTransform)
)
pp.pprint(step3info)

In [ ]:
bounds

# Step 4. Cross-section Slice Viewer

In [ ]:
print(vdvol)

In [ ]:
#pl = vedo.applications.Slicer2DPlotter(vdvol,levels=(20,60));
#pl = vedo_plotters.SimonSlicer2DPlotter(vdvol,levels=[20,60])
import vedo
vdvol = vedo.Volume(mergedvol);
pl = vedo_plotters.SimonSlicer2DPlotter(
    vdvol,
    histo_color=None,
);
pl.show(
    #viewup=[-1,0,0],
    viewup=[0,1,0],
    interactive=True,
    #axes=4
);
pl.close()


In [ ]:
pl.close()

# Step 5. Test SimpleITK Segmentation Method

In [ ]:
vdvol.dataset.GetPointData().GetScalars().GetNumberOfComponents()

In [12]:
# Get ITK image without requiring new memory
import itk
import vtk

# create itk_image
itk_image = itk.image_from_vtk_image(vdvol.dataset)

# itk_image
print(itk_image)

Image (000001F30718EC10)
  RTTI typeinfo:   class itk::Image<unsigned char,3>
  Reference Count: 1
  Modified Time: 9
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 0
  UpdateMTime: 0
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 9000, 350]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 9000, 350]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 9000, 350]
  Spacing: [0.003475, 0.02, 0.02]
  Origin: [0, 0, 0]
  Direction: 
1 0 0
0 1 0
0 0 1

  IndexToPointMatrix: 
0.003475 0 0
0 0.02 0
0 0 0.02

  PointToIndexMatrix: 
287.77 0 0
0 50 0
0 0 50

  Inverse Direction: 
1 0 0
0 1 0
0 0 1

  PixelContainer: 
    ImportImageContainer (000001F306843440)
      RTTI typeinfo:   class itk::ImportImageContainer<unsigned __int64,unsigned char>
      Re

In [13]:
itk_image.GetImageDimension()

3

In [14]:
# make simpleitk image
def itkToSimpleITK(itk_image):
    new_sitk_image = sitk.GetImageFromArray(itk.GetArrayViewFromImage(itk_image),isVector=itk_image.GetNumberOfComponentsPerPixel()>1);
    new_sitk_image.SetOrigin(tuple(itk_image.GetOrigin()))
    new_sitk_image.SetSpacing(tuple(itk_image.GetSpacing()))
    new_sitk_image.SetDirection(itk.GetArrayFromMatrix(itk_image.GetDirection()).flatten()) 
    return new_sitk_image;
simgmerged = itkToSimpleITK(itk_image);

In [15]:
# -- INITIATE SIMPLE ITK UTILITIES FROM THE NOTEBOOKS CODE
import sys
#sys.path.append('sandbox_sg/SimpleITK-Notebooks/Utilities')
sys.path.append('sandbox_sg/SimpleITK-Notebooks/Python')

#from downloaddata import fetch_data as fdata
from myshow import myshow, myshow3d

In [ ]:
# pyvista/vtk to simpleitk
# from SimpleITK.utilities.vtk import vtk2sitk, sitk2vtk
# simgmerged = vtk2sitk(mergedvol);

In [ ]:
print(simgmerged)

In [ ]:
# To visualize the labels image in RGB with needs a image with 0-255 range
sitk_filter_calc_minmax = sitk.MinimumMaximumImageFilter();

sitk_filter_calc_minmax.Execute(simgmerged);
print('Scalar Min:{:} Max:{:}'.format(
      sitk_filter_calc_minmax.GetMinimum(),
      sitk_filter_calc_minmax.GetMaximum()
));

# rescale d image to a narrower fixed range, normalize to 0-255, and cast to an unsigned integer
sitk_filter = sitk.IntensityWindowingImageFilter();
sitk_filter.SetOutputMaximum(255);
sitk_filter.SetOutputMinimum(0);
sitk_filter.SetWindowMinimum(30);
sitk_filter.SetWindowMaximum(60);
#simg_processed = sitk_filter.Execute(simgmerged);
simg_processed = sitk.Cast(sitk_filter.Execute(simgmerged), sitk.sitkUInt8)
# #simg_processed = sitk
# #img_T1_255 = sitk.Square(simgmerged);

sitk_filter_calc_minmax.Execute(simg_processed);
print('Scalar Min:{:} Max:{:}'.format(
      sitk_filter_calc_minmax.GetMinimum(),
      sitk_filter_calc_minmax.GetMaximum()
));

In [ ]:
print(simg_processed)

In [ ]:
myshow3d(simg_processed)


In [ ]:
del simgmerged
del sitk_filter
del sitk_filter_calc_minmax


In [ ]:
del octstudy

In [ ]:
del mergedvol

## Tresholding

### Basic Threshold

In [ ]:
seg = simg_processed > 200
myshow(sitk.LabelOverlay(simg_processed, seg), "Basic Thresholding")
#sitk.Show(sitk.LabelOverlay(simg_processed, seg))

### Binary Thresholding

In [ ]:
seg = sitk.BinaryThreshold(
    simgmerged, lowerThreshold=250, upperThreshold=255, insideValue=1, outsideValue=0
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged, seg), title="Binary Thresholding", dpi=10, defaultslice=110 )

### Thresholding: Otsu

In [ ]:
otsu_filter = sitk.OtsuThresholdImageFilter()
otsu_filter.SetInsideValue(0)
otsu_filter.SetOutsideValue(1)
seg = otsu_filter.Execute(simgmerged)
title="Otsu Thresholding [Threshold={:}]".format(otsu_filter.GetThreshold());

from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged, seg), title=title, dpi=10, defaultslice=110 )
# print(otsu_filter.GetThreshold())
# print(otsu_filter.GetNumberOfHistogramBins());

In [19]:
del otsu_filter

### Thresholding: Binary Thresholding On A Slice In The Middle

In [43]:
size = simgmerged.GetSize()
slice_x = slice(0,size[1])
slice_y = slice(size[1]//2-(size[1]//4) , size[1]//2+(size[1]//4) )
simgmerged_midstrip = simgmerged[:,slice_y,:]
print(simgmerged_midstrip.GetSize())
#from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
#sitk_myshow(simgmerged_midstrip)
sitk_plotters.ImageSITKSliceViewer3DJupyter(simgmerged_midstrip)

(690, 4500, 350)


TraitError: Invalid selection: value not found

In [ ]:
seg = sitk.BinaryThreshold(
    simgmerged_midstrip, lowerThreshold=250, upperThreshold=255, insideValue=1, outsideValue=0
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged_midstrip, seg), title="Binary Thresholding", dpi=10, defaultslice=110 )

interactive(children=(IntSlider(value=110, description='z', max=349), Output()), _dom_classes=('widget-interac…

In [23]:
print(simgmerged_midstrip)

Image (000001F30718D590)
  RTTI typeinfo:   class itk::Image<unsigned char,3>
  Reference Count: 1
  Modified Time: 2215
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 2203
  UpdateMTime: 2214
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 4500, 350]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 4500, 350]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 4500, 350]
  Spacing: [0.003475, 0.02, 0.02]
  Origin: [0, 45, 0]
  Direction: 
1 0 0
0 1 0
0 0 1

  IndexToPointMatrix: 
0.003475 0 0
0 0.02 0
0 0 0.02

  PointToIndexMatrix: 
287.77 0 0
0 50 0
0 0 50

  Inverse Direction: 
1 0 0
0 1 0
0 0 1

  PixelContainer: 
    ImportImageContainer (000001EEF4C31B40)
      RTTI typeinfo:   class itk::ImportImageContainer<unsigned __int64,unsigned char

In [25]:
simgmerged_midstrip.SetDirection((0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0))

In [27]:
print(simgmerged_midstrip)

Image (000001F30718D590)
  RTTI typeinfo:   class itk::Image<unsigned char,3>
  Reference Count: 1
  Modified Time: 2301
  Debug: Off
  Object Name: 
  Observers: 
    none
  Source: (none)
  Source output name: (none)
  Release Data: Off
  Data Released: False
  Global Release Data: Off
  PipelineMTime: 2203
  UpdateMTime: 2214
  RealTimeStamp: 0 seconds 
  LargestPossibleRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 4500, 350]
  BufferedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 4500, 350]
  RequestedRegion: 
    Dimension: 3
    Index: [0, 0, 0]
    Size: [690, 4500, 350]
  Spacing: [0.003475, 0.02, 0.02]
  Origin: [0, 45, 0]
  Direction: 
0 1 0
1 0 0
0 0 1

  IndexToPointMatrix: 
0 0.02 0
0.003475 0 0
0 0 0.02

  PointToIndexMatrix: 
0 287.77 0
50 0 0
0 0 50

  Inverse Direction: 
0 1 0
1 0 0
0 0 1

  PixelContainer: 
    ImportImageContainer (000001EEF4C31B40)
      RTTI typeinfo:   class itk::ImportImageContainer<unsigned __int64,unsigned char

In [26]:
seg = sitk.BinaryThreshold(
    simgmerged_midstrip, lowerThreshold=250, upperThreshold=255, insideValue=1, outsideValue=0
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged_midstrip, seg), title="Binary Thresholding", dpi=10, defaultslice=110 )

interactive(children=(IntSlider(value=110, description='z', max=349), Output()), _dom_classes=('widget-interac…

In [37]:
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

sitk_plotters.ImageSITKSliceViewer3DJupyter( simgmerged_midstrip )

NameError: name 'simgmerged_midstrip' is not defined

In [36]:
simgmerged_midstrip.GetSize()

(690, 4500, 350)

### Thresholding: Otsu On A Slice In The Middle

In [ ]:
size = simgmerged.GetSize()
slice_x = slice(0,size[1])
slice_y = slice(size[1]//2-(size[1]//4) , size[1]//2+(size[1]//4) )
simgmerged_midstrip = simgmerged[:,slice_y,:]
print(simgmerged_midstrip.GetSize())
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(simgmerged_midstrip)

(690, 4500, 350)


interactive(children=(IntSlider(value=174, description='z', max=349), Output()), _dom_classes=('widget-interac…

In [ ]:
size = simgmerged.GetSize()
slice_x = slice(0,size[1])
slice_y = slice(size[1]//2-(size[1]//4) , size[1]//2+(size[1]//4) )
print(slice_y)
simgmerged_midstrip = simgmerged[:,slice_y,:]
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
simgmerged_midstrip_array = sitk.GetArrayViewFromImage(simgmerged_midstrip)

slice(2250, 6750, None)


In [ ]:
np.sum(np.sum(simgmerged_midstrip_array,axis=0),axis=1).shape

In [ ]:
otsu_filter = sitk.OtsuThresholdImageFilter()
otsu_filter.SetInsideValue(0)
otsu_filter.SetOutsideValue(1)
otsu_filter.SetNumberOfHistogramBins(8)
#otsu_filter.
seg = otsu_filter.Execute(simgmerged_midstrip)
title="Otsu Thresholding [Threshold={:}]".format(otsu_filter.GetThreshold());

from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simgmerged_midstrip, seg), title=title, dpi=10, defaultslice=110 )
print(otsu_filter.GetThreshold())
print(otsu_filter.GetNumberOfHistogramBins());

### Thresholding: Maximum Entropy

In [ ]:
filter_threshold_maxentropy = sitk.MaximumEntropyThresholdImageFilter();
filter_threshold_maxentropy.SetInsideValue(0)
filter_threshold_maxentropy.SetOutsideValue(1)
seg = filter_threshold_maxentropy.Execute(simg_processed)
title="MaximumEntropyThresholdImageFilter [Threshold={:}]".format(filter_threshold_maxentropy.GetThreshold());
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simg_processed, seg), title=title, dpi=10, defaultslice=110 )
print(filter_threshold_maxentropy.GetThreshold())
print(filter_threshold_maxentropy.GetNumberOfHistogramBins());

In [ ]:
del filter_threshold_maxentropy

### Thresholding: Huang

In [ ]:
filter_threshold_huang = sitk.HuangThresholdImageFilter();
filter_threshold_huang.SetInsideValue(0)
filter_threshold_huang.SetOutsideValue(1)
seg = filter_threshold_huang.Execute(simg_processed)
title="HuangThresholdImageFilter [Threshold={:}]".format(filter_threshold_huang.GetThreshold());
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow( sitk.LabelOverlay(simg_processed, seg), title=title, dpi=10, defaultslice=110 )
print(filter_threshold_huang.GetThreshold())
print(filter_threshold_huang.GetNumberOfHistogramBins());

In [ ]:
del filter_threshold_huang

In [ ]:
plt.close('all')

## Region Growing Segmentation

The first step of improvement upon the naive thresholding is a class of algorithms called region growing. This includes:
<ul>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ConnectedThresholdImageFilter.html">ConnectedThreshold</a></li>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ConfidenceConnectedImageFilter.html">ConfidenceConnected</a></li>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1VectorConfidenceConnectedImageFilter.html">VectorConfidenceConnected</a></li>
  <li><a href="http://www.itk.org/Doxygen/html/classitk_1_1NeighborhoodConnectedImageFilter.html">NeighborhoodConnected</a></li>
</ul>

Earlier we used 3D Slicer to determine that index: (132,142,96) was a good seed for the left lateral ventricle.

In [ ]:
sitk.Show(simg_processed)

In [ ]:
#seed = (132, 142, 96)
seed = (222,5247,106)
seg = sitk.Image(simgmerged.GetSize(), sitk.sitkUInt8)
seg.CopyInformation(simgmerged)
seg[seed] = 1
seg = sitk.BinaryDilate(seg, [3] * 3)
#sitk_myshow(sitk.LabelOverlay(simg_processed, seg), "Initial Seed");

### Connected Threshold

In [ ]:
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point
seg = sitk.ConnectedThreshold(simgmerged, seedList=[seed], lower=225, upper=255);
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelOverlay(simgmerged, seg), "Connected Threshold");
#sitk.Show( sitk.LabelOverlay(simg_processed, seg) );

### Confidence-Connected-Threshold (automatically set a threshold based on neighborhood)

In [ ]:
# Improving upon this is the ConfidenceConnected filter, which uses the initial seed or current segmentation to estimate the threshold range.
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point
seg = sitk.ConfidenceConnected(
    simgmerged,
    seedList=[seed],
    numberOfIterations=1,
    multiplier=1.0,
    initialNeighborhoodRadius=1,
    replaceValue=1,
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelOverlay(simgmerged, seg), "ConfidenceConnected");
#sitk.Show(sitk.LabelOverlay(simgmerged, seg))

In [ ]:
# query this segmentation
filter_label_statistics = sitk.LabelStatisticsImageFilter();
filter_label_statistics.Execute(simgmerged,seg);


In [ ]:

# Get the label values that have statistics
labels = filter_label_statistics.GetLabels()
print(f"Labels: {labels}")

# Iterate over each label and print statistics
for label in labels:
    print(f"Statistics for label: {label}");
    print(f"  Mean: {filter_label_statistics.GetMean(label)}");
    print(f"  Minimum: {filter_label_statistics.GetMinimum(label)}");
    print(f"  Maximum: {filter_label_statistics.GetMaximum(label)}");
    print(f"  Sum: {filter_label_statistics.GetSum(label)}");
    print(f"  Variance: {filter_label_statistics.GetVariance(label)}");
    print(f"  Standard Deviation: {filter_label_statistics.GetSigma(label)}");
    print(f"  BoundingBox: {str(filter_label_statistics.GetBoundingBox(label))}");
    print(f"  Region: {str(filter_label_statistics.GetRegion(label))}");
    #print(f"  Number of pixels: {filter_label_statistics.GetNumberOfPixels(label)}")

# Test
#filter_label_statistics.

## Fast Marching Segmentation

The FastMarchingImageFilter implements a fast marching solution to a simple level set evolution problem (eikonal equation). In this example, the speed term used in the differential equation is provided in the form of an image. The speed image is based on the gradient magnitude and mapped with the bounded reciprocal $1/(1+x)$.


In [ ]:
#seed = (132, 142, 96)
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point
feature_img = sitk.GradientMagnitudeRecursiveGaussian(simg_processed, sigma=0.5)
speed_img = sitk.BoundedReciprocal(
    feature_img
)  # This is parameter free unlike the Sigmoid
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(speed_img)

In [ ]:
del feature_img
del speed_img

## Level-Set Segmentation

There are a variety of level-set based segmentation filter available in ITK:
<ul>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1GeodesicActiveContourLevelSetImageFilter.html">GeodesicActiveContour</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ShapeDetectionLevelSetImageFilter.html">ShapeDetection</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ThresholdSegmentationLevelSetImageFilter.html">ThresholdSegmentation</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1LaplacianSegmentationLevelSetImageFilter.html">LaplacianSegmentation</a></li>
<li><a href="http://www.itk.org/Doxygen/html/classitk_1_1ScalarChanAndVeseDenseLevelSetImageFilter.html">ScalarChanAndVese</a></li>
</ul>

There is also a <a href="http://www.itk.org/Doxygen/html/group__ITKLevelSetsv4.html">modular Level-set framework</a> which allows composition of terms and easy extension in C++.




First we create a label image from our seed.

In [ ]:
seed = (222,5247,106); # found using fiji, coordinate on top surface of the device at a particular point

seg = sitk.Image(simg_processed.GetSize(), sitk.sitkUInt8)
seg.CopyInformation(simg_processed)
seg[seed] = 1
seg = sitk.BinaryDilate(seg, [3] * 3)

Use the seed to estimate a reasonable threshold range.

In [ ]:
stats = sitk.LabelStatisticsImageFilter();
stats.Execute(simg_processed, seg);

factor = 1.0;
lower_threshold = stats.GetMean(1) - factor * stats.GetSigma(1);
#upper_threshold = stats.GetMean(1) + factor * stats.GetSigma(1)
lower_threshold = 100;
upper_threshold = 255;
print(lower_threshold, upper_threshold);

In [ ]:
init_ls = sitk.SignedMaurerDistanceMap(seg, insideIsPositive=True, useImageSpacing=True)

In [ ]:
lsFilter = sitk.ThresholdSegmentationLevelSetImageFilter()
lsFilter.SetLowerThreshold(lower_threshold)
lsFilter.SetUpperThreshold(upper_threshold)
lsFilter.SetMaximumRMSError(0.0002)
lsFilter.SetNumberOfIterations(2)
lsFilter.SetCurvatureScaling(2.0)
lsFilter.SetPropagationScaling(1)
lsFilter.ReverseExpansionDirectionOn()
ls = lsFilter.Execute(init_ls, sitk.Cast(simg_processed, sitk.sitkFloat32))
print(lsFilter)

In [ ]:
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelOverlay(simg_processed, ls > 0))

## Gradient Watersheds Segmentation

In [ ]:
sigma = simgmerged.GetSpacing()[0]
level = 4

In [ ]:
feature_img = sitk.GradientMagnitude(simgmerged);
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(feature_img);

In [ ]:
ws_img = sitk.MorphologicalWatershed(
    feature_img, level=0, markWatershedLine=True, fullyConnected=False
)
from naatos_oct_tools.plotters.sitk_plotters import sitk_myshow
sitk_myshow(sitk.LabelToRGB(ws_img), "Watershed Over Segmentation")

# Test Line Extraction

In [ ]:
#pvvol = octstudy.octdatalist[10].pvvol;
pvvol = mergedvol;
pl = pv.Plotter(notebook=False);


#plt.add_volume(pvvol,opacity=0.1);
#pl.add_volume_clip_plane(pvvol,normal='x',assign_to_axis='x',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='x',assign_to_axis='x',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='y',assign_to_axis='y',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='y',cmap='grey',clim=[40,55]);
pl.add_mesh_slice(pvvol,normal='z',assign_to_axis='z',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice(pvvol,normal='z',cmap='grey',clim=[40,55]);
#pl.add_mesh_slice_orthogonal(pvvol,cmap='grey',clim=[40,55]);

#pl.add_volume(pvvol,opacity=0.2);
# def callback(normal, origin):
#     slc = pvvol.slice(normal=normal, origin=origin)
#     origin = list(origin)
#     origin[2] = slc.bounds[5]
#     peak_plane = pv.Plane(
#         center=origin,
#         direction=[0, 0, 1],
#         i_size=20,
#         j_size=20,
#     )
#     _ = pl.add_mesh(
#         peak_plane, name="Peak", color='red', opacity=0.4
#     )
# _ = pl.add_plane_widget(callback, normal_rotation=False)

#pl.add_axes();

# def move_center(pointa, pointb):
#     center = (np.array(pointa) + np.array(pointb)) / 2
#     normal = np.array(pointa) - np.array(pointb)
#     single_slc = pvvol.slice(normal=normal, origin=center)

#     _ = pl.add_mesh(single_slc, name="slc")

# _ = pl.add_line_widget(callback=move_center, use_vertices=True)
pl.add_box_axes();
pl.show_bounds();

pl.show();

In [ ]:
p.plane_sliced_meshes[0].plot(notebook=False)

In [ ]:
from SimpleITK.utilities.vtk import vtk2sitk
simgmerged = vtk2sitk(mergedvol)

In [ ]:
plt = pv.Plotter(notebook=False);
plt.add_volume(mergedvol);
plt.show(jupyter_backend='none')


In [ ]:
import cv2